Testing notebook for fitting a global propensity model and matching cells. Replaced by `global_psm.ipynb` and `match_cells.ipynb`.

In [1]:
from pathlib import Path
import sys
import os
import ee
import geemap

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    COUNTRIES_ASSET_ID,
    PAS_ASSET_ID,
    OECMS_ASSET_ID,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    TREATMENT_CELLS,
    CONTROL_CELLS,
    GPD_CRS_METERS,
)

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

from absolute_effectiveness.site_selector import SiteSelector

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

Part 1: Fit a global propensity model

In [ ]:
# Select AOI (country/region, eventually scale up to the whole world)

country_name = "Russia"

countries = ee.FeatureCollection(COUNTRIES_ASSET_ID)
aoi = countries.filter(ee.Filter.eq("country_na", country_name))

In [ ]:
# Import PAs and filter to AOI

PAs = ee.FeatureCollection(PAS_ASSET_ID).filterBounds(aoi)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID).filterBounds(aoi)

all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)


EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

protected_mask = (
    ee.Image.constant(0)
    .rename("protected")
    .paint(featureCollection=all_PAs, color=1)
    .reproject(crs=EE_CRS_1km)
)

In [ ]:
# Import covariate datasets

elevation = ee.ImageCollection("COPERNICUS/DEM/GLO30").filterBounds(aoi).select("DEM")
slope = elevation.map(lambda tile: ee.Terrain.slope(tile))
treecover2000 = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("treecover2000")
# Travel time in minutes to the nearest densely-populated area
travel_time = ee.Image("projects/malariaatlasproject/assets/accessibility/accessibility_to_cities/2015_v1_0").select("accessibility").rename("travel_time")
log_pop_density = (ee.Image('JRC/GHSL/P2023A/GHS_POP/2000')
               .select("population_count")
               .add(1) # handles zeros for log transform
               .log()
               .rename("log_pop_density"))

elevation = elevation.mosaic().rename("elevation")
slope = slope.mosaic()

# Resample covariates to 1km resolution

def resample_covariates(covariate):
    return (
        covariate.setDefaultProjection(EE_CRS_1km)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
        .reproject(protected_mask.projection())
    )

elevation = resample_covariates(elevation)
slope = resample_covariates(slope)
treecover2000 = resample_covariates(treecover2000)
travel_time = resample_covariates(travel_time)
log_pop_density = resample_covariates(log_pop_density)

# Stack covariates in a single multi-band image
covariates = protected_mask.addBands(elevation).addBands(slope).addBands(treecover2000).addBands(travel_time).addBands(log_pop_density)

In [ ]:
# Apply land mask to avoid sampling permanent water / ocean

land_mask = (
    ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)

covariates = covariates.updateMask(land_mask)

In [ ]:
# Proportionally allocate samples across biomes and protected / unprotected areas

# Construct categorical biome raster
biome_fc = (ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")
    .filterBounds(aoi)
    .map(lambda f: f.set("BIOME_NUM", ee.Number(f.get("BIOME_NUM")).int()))
    .map(lambda f: f.simplify(10000)) # simplify geometries
)

biome = (
    ee.Image(0)
    .paint(
        featureCollection=biome_fc,
        color="BIOME_NUM"
    )
    .rename("biome")
    .reproject(protected_mask.projection())
    .toInt()
)

# Create combined strata raster
strata = protected_mask.multiply(20).add(biome).rename("strata").toInt()
covariates = covariates.addBands(strata).addBands(biome)

# Compute area of each biome to determine proportional allocation
# (this is in square degrees so approximate, but sufficent for rough proportions)
rows = biome_fc.select(["BIOME_NUM", "SHAPE_AREA"]).getInfo()["features"]
biome_df = pd.DataFrame([f["properties"] for f in rows])
biome_area_by_num = biome_df.groupby("BIOME_NUM")["SHAPE_AREA"].sum()

print(biome_area_by_num)

# # Calculate number of points to sample for each stratum
total_points = 10000
min_per_stratum = 20
total_area = biome_area_by_num.sum()

# Define desired ratio of unprotected to protected samples (currently 2:1)
treatment_budget = total_points // 3
control_budget = total_points * 2 // 3

class_values = []
class_points = []

for protected, budget in [(0, control_budget), (1, treatment_budget)]:
    for biome_num, area in biome_area_by_num.items():
        stratum = protected * 20 + biome_num
        proportional = int((area / total_area) * budget)
        n = max(proportional, min_per_stratum)
        class_values.append(stratum)
        class_points.append(n)

print("Class values: ", class_values)
print("Class points: ", class_points)

In [ ]:
# Take a stratified sample of covariates

# Sample pixel locations from strata band only
sample_pixels = strata.stratifiedSample(
    region=aoi.geometry(),
    scale=PSM_CELL_SIZE,
    projection=EE_CRS_1km,
    seed=42,
    numPoints=1,
    classBand="strata",
    classValues=class_values,
    classPoints=class_points,
    dropNulls=True,
    geometries=True,
)

# Extract covariate values at sampled locations
sample_pixels = covariates.select(
    ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected"]
).sampleRegions(
    collection=sample_pixels,
    scale=PSM_CELL_SIZE,
    projection=EE_CRS_1km,
    geometries=True,
)

# Export samples to asset
task = ee.batch.Export.table.toAsset(
    collection=sample_pixels,
    description="psm_training_samples",
    assetId=f"projects/{PROJECT}/assets/TPAE/psm_training_samples_{country_name}_{total_points}",
)
task.start()
print("Export started:", task.status())

In [ ]:
# Import samples

samples_fc = ee.FeatureCollection(f"projects/{PROJECT}/assets/TPAE/psm_training_samples_{country_name}_{total_points}")
samples_gdf = geemap.ee_to_gdf(samples_fc).to_crs(GPD_CRS_METERS)

print(samples_gdf.head())
print(len(samples_gdf))

In [ ]:
# Thin samples so they are at least 3km apart to avoid spatial autocorrelation

# Shuffle samples first to avoid bias in which points get dropped
samples_gdf_shuffled = samples_gdf.sample(frac=1, random_state=42).reset_index(drop=True)

kept = []
kept_geoms = []
for idx, row in samples_gdf_shuffled.iterrows():
    pt = row.geometry
    if all(pt.distance(k) >= 3000 for k in kept_geoms):
        kept.append(idx)
        kept_geoms.append(pt)

samples_df_thinned = samples_gdf_shuffled.loc[kept].reset_index(drop=True)

print("Number of samples after thinning: ", len(samples_df_thinned))
print(samples_df_thinned.head())

In [ ]:
# Use training data to fit a propensity model (logistic regression)
X = samples_df_thinned[["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]]
y = samples_df_thinned["protected"]
model = Pipeline(
    [("scaler", StandardScaler()), ("logit", LogisticRegression(max_iter=1000))]
)
model.fit(X, y)

# Examine coefficients to confirm that the data is reproducing the expected PA location bias
predictors = X.columns
logit = model.named_steps["logit"]
coef_df = pd.DataFrame({"variable": predictors, "coefficient": logit.coef_[0]})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df["effect_on_protection"] = coef_df["coefficient"].apply(
    lambda x: "positive" if x > 0 else "negative"
)
print("\nCoefficient summary:")
print(coef_df.sort_values("abs_coefficient", ascending=False))

In [ ]:
from utils.variables import BIOME_PALETTE

Map = geemap.Map()

Map.add_basemap("CartoDB.Positron")

Map.addLayer(aoi, {}, "AOI")
Map.centerObject(aoi)

# Propensity Model
Map.addLayer(
    covariates.select("protected").clip(aoi),
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "Protected",
)
Map.addLayer(
    covariates.select("elevation"),
    {"min": 0, "max": 800, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation",
)
Map.addLayer(
    covariates.select("slope"),
    {"min": 0, "max": 12, "palette": ["blue", "green", "yellow", "red"]},
    "Slope",
)
Map.addLayer(
    covariates.select("treecover2000"),
    {"min": 0, "max": 100, "palette": ["white", "green"]},
    "Tree Cover 2000",
)
Map.addLayer(
    covariates.select("travel_time"),
    {"min": 0, "max": 1200, "palette": ["blue", "green", "yellow", "red"]},
    "Travel Time",
)
Map.addLayer(
    covariates.select("log_pop_density"),
    {"min": 0, "max": 6, "palette": ["blue", "green", "yellow", "red"]},
    "Population Density",
)
Map.addLayer(covariates.select("biome"), {"min": 1, "max": 14, "palette": BIOME_PALETTE}, "Biome")
# Map.addLayer(land_mask, {}, "Land Mask")
Map.addLayer(samples_fc, {"color": "red"}, "Sample Pixels")

# # Cell Matching
# Map.addLayer(site_geom, {"color": "red"}, "Site")
# Map.addLayer(grid_fc, {}, "Grid")
# Map.addLayer(matched_grids, {}, "Matched Cells")

Map

Part 2: Match treatment and control cells

In [ ]:
# Load valid grid cells for a given PA

# Select PA
PA_ID = "1543"
site_id = 1543
test_sites = site_selector.get_test_sites()
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Import treatment and control cells and filter to PA
treatment_cells = gpd.read_parquet(TREATMENT_CELLS).to_crs(epsg=4326)
treatment_cells = treatment_cells[treatment_cells["WDPA_PID"] == PA_ID]
control_cells = gpd.read_parquet(CONTROL_CELLS).to_crs(epsg=4326)
control_cells = control_cells[control_cells["WDPA_PID"] == PA_ID]

# Convert to ee.FeatureCollection
treatment_fc = geemap.geopandas_to_ee(treatment_cells)
control_fc = geemap.geopandas_to_ee(control_cells)
all_cells = ee.FeatureCollection([treatment_fc, control_fc]).flatten()

# Add a unique cell_ID to each grid cell
cell_IDs = ee.List.sequence(0, all_cells.size().getInfo() - 1)
featureList = all_cells.toList(all_cells.size())
grid_fc = ee.FeatureCollection(
    cell_IDs.map(
        lambda cell_ID: ee.Feature(featureList.get(cell_ID)).set(
            {"cell_ID": cell_ID, "label": None}
        )
    )
)

print(grid_fc.size().getInfo())

In [ ]:
# Aggregate covariates within grid cells
# will need to use mode reducer for categorical covariates

grid_fc = (
    covariates.select("elevation", "slope", "treecover2000", "travel_time", "log_pop_density")
    .reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.mean(),
        scale=PSM_CELL_SIZE,
        crs=EE_CRS_1km,
    )
    .select("cell_ID", "elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected")
)

# print(grid_fc.first().getInfo())

In [ ]:
# Apply global propensity model from Part 1 to predict propensity scores for each grid cell

grid_list = grid_fc.getInfo()["features"]
grid_df = pd.DataFrame([feature["properties"] for feature in grid_list])

X = grid_df[["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]]
grid_df["propensity_score"] = model.predict_proba(X)[:, 1]

print(grid_df.head())

In [ ]:
# Match each treatment cell to up to 4 control cells
# Re-using control cells is ok

# Split treatment and control
treat_df = grid_df[grid_df["protected"] == 1].copy().reset_index(drop=True)
control_df = grid_df[grid_df["protected"] == 0].copy().reset_index(drop=True)

# Convert to 2D array
X_control = control_df[["propensity_score"]].to_numpy()
X_treat = treat_df[["propensity_score"]].to_numpy()

# Fit nearest-neighbor search on control cells
nn = NearestNeighbors(
    n_neighbors=4,
    metric="euclidean",  # in 1D, this is just absolute difference
)
nn.fit(X_control)

# Find the 4 nearest control cells for each treatment cell
distances, indices = nn.kneighbors(X_treat)

# Build a long-form match table
matches = []
caliper = 0.2

for i in range(len(treat_df)):
    treat_row = treat_df.iloc[i]
    treat_id = treat_row["cell_ID"]
    treat_score = treat_row["propensity_score"]

    for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
        # Apply caliper
        if dist <= caliper:
            control_row = control_df.iloc[j]

            matches.append(
                {
                    "treat_cell_id": treat_id,
                    "control_cell_id": control_row["cell_ID"],
                    "treat_score": treat_score,
                    "control_score": control_row["propensity_score"],
                    "ps_distance": float(dist),
                    "match_rank": rank,
                }
            )

match_df = pd.DataFrame(matches).sort_values(by="treat_cell_id")

print(match_df.head())
print(f"\nNumber of matched treatment cells: {match_df['treat_cell_id'].nunique()}")
print(f"Total matched pairs: {len(match_df)}")
unmatched_treat_ids = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"Unmatched treatment cells: {len(unmatched_treat_ids)}")

In [ ]:
# Filter grid cells to only valid matches

valid_ids = pd.concat([match_df["treat_cell_id"], match_df["control_cell_id"]]).unique()
valid_ids = ee.List(valid_ids.astype(int).tolist())
matched_grids = grid_fc.filter(ee.Filter.inList("cell_ID", valid_ids))

In [ ]:
# Save matched_grids and propensity match pairs to parquet
matched_grids_gdf = geemap.ee_to_gdf(matched_grids)
matched_grids_gdf.to_parquet(f"data/matched_grids_{PA_ID}.parquet")
match_df.to_parquet(f"data/match_table_{PA_ID}.parquet", index=False)